# ARCHIVED — the difficulty sweep

**Do not treat this as pipeline code.** It is the exploratory notebook that established
three constraints on the method, all of which are now stated as facts in the main README:

- **RadEdit is architecturally locked to 512 px.** Generation at 768 and 1024 fails with a
  tensor shape mismatch (96 vs 64) — the UNet carries 64x64 latent structure. The smallest
  workable mask is 47 px, about 32 mm, so every lesion this pipeline can produce is a
  **mass, not a nodule**.
- **Prompt text does not control lesion type.** Across nodule, mass, consolidation,
  airspace opacity and ground-glass opacity, detection ran 0.899 to 0.954 — *upward* — on
  visually indistinguishable images.
- **Conspicuity is a real axis, but only post-hoc.** Scaling a generated lesion back toward
  the background gives a clean monotonic response, with detection collapsing between
  CNR 0.66 and 0.52. Those are attenuated generations, not independently generated subtle
  lesions, and must be reported as a conspicuity axis rather than as hard cases.

Everything here ran on a **single chest (c0221)**, which later turned out to be one where
RadEdit edits unusually weakly. The 40% failure rate measured here does not reproduce
across chests. Kept because the three constraints above do.

In [ ]:
!pip -q install -U diffusers transformers accelerate huggingface-hub SimpleITK

from google.colab import userdata, drive
from huggingface_hub import login

# Colab: key icon in the left sidebar -> add secret named HF_TOKEN -> toggle
# "Notebook access" on. Token needs read scope, and you must have accepted the
# gated-model terms at huggingface.co/microsoft/radedit while logged in.
login(token=userdata.get('HF_TOKEN'))
drive.mount('/content/drive')
print('authenticated')

In [ ]:
import torch
from transformers import AutoModel, AutoTokenizer
from diffusers import (AutoencoderKL, DDIMScheduler, DiffusionPipeline,
                       StableDiffusionPipeline, UNet2DConditionModel)

unet = UNet2DConditionModel.from_pretrained('microsoft/radedit', subfolder='unet')
vae  = AutoencoderKL.from_pretrained('stabilityai/sdxl-vae')
text_encoder = AutoModel.from_pretrained('microsoft/BiomedVLP-BioViL-T', trust_remote_code=True)
tokenizer    = AutoTokenizer.from_pretrained('microsoft/BiomedVLP-BioViL-T',
                                             model_max_length=128, trust_remote_code=True)
scheduler = DDIMScheduler(beta_schedule='linear', clip_sample=False,
                          prediction_type='epsilon', timestep_spacing='trailing',
                          steps_offset=1)

gen_pipe = StableDiffusionPipeline(
    vae=vae, text_encoder=text_encoder, tokenizer=tokenizer, unet=unet,
    scheduler=scheduler, safety_checker=None, requires_safety_checker=False,
    feature_extractor=None).to('cuda')

radedit = DiffusionPipeline.from_pipe(gen_pipe, custom_pipeline='microsoft/radedit', trust_remote_code=True)
print('radedit loaded')

In [ ]:
import glob, numpy as np, pandas as pd, cv2, SimpleITK as sitk
from pathlib import Path
from PIL import Image, ImageDraw, ImageOps
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.transforms import functional as TF
import matplotlib.pyplot as plt, matplotlib.patches as patches

OUT = Path('/content/out'); (OUT/'img').mkdir(parents=True, exist_ok=True)
DEVICE, DET_SIZE, SCORE_MIN = 'cuda', 800, 0.05

ck = glob.glob('/content/drive/MyDrive/**/baseline1_checkpoint.pth', recursive=True)
assert ck, 'checkpoint not found'
det = torchvision.models.detection.fasterrcnn_resnet50_fpn(
    weights=None, weights_backbone=None, box_score_thresh=0.0, box_detections_per_img=300)
det.roi_heads.box_predictor = FastRCNNPredictor(
    det.roi_heads.box_predictor.cls_score.in_features, 2)
st = torch.load(ck[0], map_location=DEVICE)
det.load_state_dict(st['model'] if 'model' in st else st)
det = det.eval().to(DEVICE)
print('detector loaded from', ck[0])

@torch.no_grad()
def detect(pil, size=DET_SIZE):
    im = pil.convert('RGB').resize((size, size), Image.LANCZOS)
    o = det([TF.to_tensor(im).to(DEVICE)])[0]
    return o['boxes'].cpu().numpy()/size, o['scores'].cpu().numpy()

def centre_in(b, t):
    cx, cy = (b[0]+b[2])/2, (b[1]+b[3])/2
    return t[0] <= cx <= t[2] and t[1] <= cy <= t[3]

def best_at(pil, target):
    b, s = detect(pil)
    k = s >= SCORE_MIN
    return float(max((ss for bb, ss in zip(b[k], s[k]) if centre_in(bb, target)), default=0.0))

In [ ]:
MHA = Path('/content/drive/MyDrive/Algoverse/Misc/node21/images/c0221.mha')
POS = (0.32, 0.40)      # fractions of the image -- never pixels

def prep(path, size=512):
    a = sitk.GetArrayFromImage(sitk.ReadImage(str(path))).astype(np.float32)
    a = a.squeeze() if a.ndim == 3 else a
    h, w = a.shape; s = min(h, w)
    a = a[(h-s)//2:(h-s)//2+s, (w-s)//2:(w-s)//2+s]        # crop, never squash
    lo, hi = np.percentile(a, [1, 99])
    a = np.clip((a-lo)/(hi-lo+1e-8), 0, 1)
    a = cv2.createCLAHE(2.0, (8,8)).apply((a*255).astype(np.uint8)).astype(np.float32)/255
    a = np.clip(cv2.resize(a, (size,size), interpolation=cv2.INTER_AREA), 0, 1)
    return Image.fromarray((a*255).astype(np.uint8)).convert('RGB')

def make_mask(cx, cy, box_frac, pad_px, size):
    half = box_frac*size/2
    x0, y0 = cx*size-half-pad_px, cy*size-half-pad_px
    x1, y1 = cx*size+half+pad_px, cy*size+half+pad_px
    assert 0 <= x0 < x1 <= size and 0 <= y0 < y1 <= size, 'mask does not fit'
    m = Image.new('L', (size,size), 0)
    ImageDraw.Draw(m).ellipse([x0,y0,x1,y1], fill=255)
    return m, (x0/size, y0/size, x1/size, y1/size), round(x1-x0, 1)

def generate(bg, prompt, mask, skip=0.3, g=7.5, steps=200, seed=42):
    torch.manual_seed(seed)
    return radedit(prompt, weights=[g], image=bg, edit_mask=mask,
                   keep_mask=ImageOps.invert(mask), num_inference_steps=steps,
                   invert_prompt='', skip_ratio=skip, output_type='pil')[0]

def edit_inside(bg, ed, mask):
    b = np.asarray(bg.convert('L'), np.float32); e = np.asarray(ed.convert('L'), np.float32)
    return float(np.abs(e-b)[np.asarray(mask, bool)].mean())

bg512 = prep(MHA, 512)
bg512.save(OUT/'img'/'background_512.png')
plt.imshow(bg512, cmap='gray'); plt.axis('off'); plt.show()

In [ ]:
mask512, target512, px512 = make_mask(*POS, 0.05, 24, 512)
assert px512 >= 47, 'below the 47px latent floor'

ref = generate(bg512, 'Right upper lobe pulmonary nodule', mask512)
ref.save(OUT/'img'/'reference.png'); mask512.save(OUT/'img'/'reference_mask.png')

ref_det = best_at(ref, target512)
bg_det  = best_at(bg512, target512)
print(f'mask {px512}px ({px512/512:.1%} of width)  '
      f'edit_inside {edit_inside(bg512, ref, mask512):.2f}')
print(f'detector: edited {ref_det:.3f}   background {bg_det:.3f}')
assert ref_det > 0.5, 'reference did not paint -- fix this before running the tests'

In [ ]:
def blend(bg, ed, a):
    b = np.asarray(bg.convert('RGB'), np.float32); e = np.asarray(ed.convert('RGB'), np.float32)
    return Image.fromarray(np.clip(b + a*(e-b), 0, 255).astype(np.uint8))

rows = []
for a in [1.0, 0.8, 0.6, 0.5, 0.4, 0.3, 0.2, 0.15, 0.1, 0.05]:
    im = blend(bg512, ref, a); im.save(OUT/'img'/f'alpha_{a:.2f}.png')
    rows.append(dict(test='A_conspicuity', alpha=a,
                     contrast=round(edit_inside(bg512, im, mask512), 2),
                     det=round(best_at(im, target512), 3)))
    print(f'α={a:<5} contrast {rows[-1]["contrast"]:6.2f}  det {rows[-1]["det"]:.3f}')

A = pd.DataFrame(rows)
mid = A[(A.det > 0.1) & (A.det < 0.7)]
print(f'\nintermediate difficulty at α = {list(mid.alpha)}' if len(mid)
      else '\nno intermediate band -- detection is a cliff, not a slope')

In [ ]:
rows, imgs = [], {}
for a in [1.00, 0.95, 0.90, 0.85, 0.80, 0.75, 0.70, 0.65, 0.60]:
    im = blend(bg512, ref, a); imgs[a] = im
    im.save(OUT/'img'/f'fine_alpha_{a:.2f}.png')
    rows.append(dict(alpha=a, contrast=round(edit_inside(bg512, im, mask512), 2),
                     det=round(best_at(im, target512), 3)))
    print(f'α={a:<5} contrast {rows[-1]["contrast"]:6.2f}  det {rows[-1]["det"]:.3f}')

A2 = pd.DataFrame(rows); A2.to_csv(OUT/'fine_alpha.csv', index=False)

# crop tight on the lesion -- at full size you cannot judge visibility
x0, y0, x1, y1 = [int(v*512) for v in target512]
pad = 40
fig, ax = plt.subplots(2, len(imgs), figsize=(2.1*len(imgs), 4.6))
for j, (a, im) in enumerate(imgs.items()):
    ax[0, j].imshow(im, cmap='gray'); ax[0, j].axis('off')
    ax[0, j].add_patch(patches.Circle((POS[0]*512, POS[1]*512), px512/2,
                                      fill=False, edgecolor='lime', lw=1.2))
    ax[1, j].imshow(im.crop((x0-pad, y0-pad, x1+pad, y1+pad)), cmap='gray'); ax[1, j].axis('off')
    ax[0, j].set_title(f'α={a}\ndet {A2[A2.alpha==a].det.iloc[0]:.3f}', fontsize=8)
plt.suptitle('bottom row: can YOU see a lesion? that is what separates hard from absent', fontsize=9)
plt.tight_layout(); plt.savefig(OUT/'fine_alpha_grid.png', dpi=130); plt.show()

In [ ]:
stab = []
for seed in [42, 7, 1234]:
    r = generate(bg512, 'Right upper lobe pulmonary nodule', mask512, seed=seed)
    for a in [1.0, 0.9, 0.85, 0.8, 0.75, 0.7]:
        im = blend(bg512, r, a)
        stab.append(dict(seed=seed, alpha=a,
                         contrast=round(edit_inside(bg512, im, mask512), 2),
                         det=round(best_at(im, target512), 3)))

S = pd.DataFrame(stab); S.to_csv(OUT/'alpha_stability.csv', index=False)
print(S.pivot(index='alpha', columns='seed', values='det').to_string())
print('\nspread across seeds at each alpha:')
print(S.groupby('alpha').det.agg(['mean','std','min','max']).round(3).to_string())

In [ ]:
rows = []
for size in [512, 768, 1024]:
    try:
        bg = prep(MHA, size)
        m, t, px = make_mask(*POS, 0.05*512/size, 24, size)   # hold px roughly constant
        if px < 47:
            m, t, px = make_mask(*POS, 0.01, 24, size)
        ed = generate(bg, 'Right upper lobe pulmonary nodule', m)
        ed.save(OUT/'img'/f'size_{size}.png')
        rows.append(dict(test='B_size', canvas=size, mask_px=px,
                         mask_frac=round(px/size, 4),
                         approx_mm=round(350*px/size, 1),      # ~35cm chest width
                         edit_inside=round(edit_inside(bg, ed, m), 2),
                         det=round(best_at(ed, t), 3)))
        print(rows[-1])
    except Exception as e:
        print(f'{size}: FAILED -- {type(e).__name__}: {e}')

B = pd.DataFrame(rows)

In [ ]:
LADDER = ['Right upper lobe pulmonary nodule',
          'Right upper lobe pulmonary mass',
          'Right upper lobe focal consolidation',
          'Right upper lobe airspace opacity',
          'Right upper lobe ground-glass opacity',
          'Subtle hazy opacity in the right upper lobe']

rows, imgs = [], {}
for p in LADDER:                       # anatomical location in ALL of them -- 3-word
    ed = generate(bg512, p, mask512)   # prompts painted nothing, 0/5 seeds
    ed.save(OUT/'img'/f'class_{p[:22].replace(" ","_")}.png'); imgs[p] = ed
    rows.append(dict(test='C_class', prompt=p,
                     edit_inside=round(edit_inside(bg512, ed, mask512), 2),
                     det=round(best_at(ed, target512), 3)))
    print(f'{p:<46} edit {rows[-1]["edit_inside"]:6.2f}  det {rows[-1]["det"]:.3f}')

C = pd.DataFrame(rows)

fig, ax = plt.subplots(1, len(imgs), figsize=(3*len(imgs), 3.4))
for a_, (p, im) in zip(ax, imgs.items()):
    a_.imshow(im, cmap='gray')
    a_.add_patch(patches.Circle((POS[0]*512, POS[1]*512), px512/2,
                                fill=False, edgecolor='lime', lw=1.5))
    a_.set_title(f'{p.split()[-2]} {p.split()[-1]}\ndet {C[C.prompt==p].det.iloc[0]:.3f}',
                 fontsize=8); a_.axis('off')
plt.suptitle('low score only counts as a HARD case if you can still see a finding', fontsize=9)
plt.tight_layout(); plt.savefig(OUT/'class_ladder.png', dpi=110); plt.show()

In [ ]:
import shutil
res = pd.concat([A, B, C], ignore_index=True)
res.to_csv(OUT/'difficulty_check.csv', index=False)

band = res[(res.det > 0.1) & (res.det < 0.7)]
print(f'\nconfigurations landing in 0.1-0.7: {len(band)} of {len(res)}')
if len(band):
    print(band.to_string(index=False))
    print('\n-> a difficulty axis exists. Route 3 is back, built on whichever test produced it.')
else:
    print('\n-> nothing intermediate across conspicuity, size and semantic class.')
    print('   Route 3 closes on evidence rather than on assumption. Write Route 2.')

z = shutil.make_archive('/content/difficulty_check', 'zip', OUT)
from google.colab import files; files.download(z)

In [ ]:
SEEDS = list(range(20))
rows, gens = [], {}
for s in SEEDS:
    g = generate(bg512, 'Right upper lobe pulmonary nodule', mask512, seed=s)
    gens[s] = g; g.save(OUT/'img'/f'seed_{s:02d}.png')
    rows.append(dict(seed=s, edit_inside=round(edit_inside(bg512, g, mask512), 2),
                     det=round(best_at(g, target512), 3)))
    print(f'seed {s:>3}  edit {rows[-1]["edit_inside"]:6.2f}  det {rows[-1]["det"]:.3f}')

SW = pd.DataFrame(rows).sort_values('det'); SW.to_csv(OUT/'seed_sweep.csv', index=False)
print('\n', SW.det.describe().round(3).to_string())
print(f'\nbelow 0.3: {(SW.det<0.3).sum()}   0.3-0.7: {SW.det.between(0.3,0.7).sum()}   '
      f'above 0.7: {(SW.det>0.7).sum()}')
print(f'correlation edit_inside vs det: r = {SW.edit_inside.corr(SW.det):+.3f}')

x0, y0, x1, y1 = [int(v*512) for v in target512]; pad = 45
fig, ax = plt.subplots(2, 10, figsize=(20, 4.4))
for a_, (_, r) in zip(ax.ravel(), SW.iterrows()):
    a_.imshow(gens[r.seed].crop((x0-pad, y0-pad, x1+pad, y1+pad)), cmap='gray')
    a_.set_title(f's{int(r.seed)}  det {r.det:.2f}\nedit {r.edit_inside:.1f}', fontsize=7)
    a_.axis('off')
plt.suptitle('sorted by detection, lowest first. Which of the left-hand ones contain a visible lesion?',
             fontsize=10)
plt.tight_layout(); plt.savefig(OUT/'seed_sweep_grid.png', dpi=130); plt.show()

In [ ]:
Y, X = np.mgrid[0:512, 0:512]
_d = np.sqrt((X-POS[0]*512)**2 + (Y-POS[1]*512)**2)
CORE = _d <= px512/2*0.7
RING = (_d > px512/2*1.15) & (_d <= px512/2*1.8)
BGN  = np.asarray(bg512.convert('L'), np.float32)
NOISE, C0 = BGN[RING].std(), BGN[CORE].mean() - BGN[RING].mean()

def cnr(pil):
    a = np.asarray(pil.convert('L'), np.float32)
    return round(float(((a[CORE].mean()-a[RING].mean()) - C0) / NOISE), 3)

SW['cnr'] = [cnr(gens[s]) for s in SW.seed]
SW.to_csv(OUT/'seed_sweep.csv', index=False)
print(SW.sort_values('det').to_string(index=False))
print(f'\nCNR vs det          r = {SW.cnr.corr(SW.det):+.3f}')
print(f'edit_inside vs det  r = {SW.edit_inside.corr(SW.det):+.3f}')

# the five zeros against the five best, image and difference map
lo = SW.nsmallest(5, 'det').seed.tolist()
hi = SW.nlargest(5, 'det').seed.tolist()
pad = int(px512/2*1.9)
cx_, cy_ = int(POS[0]*512), int(POS[1]*512)
box = (cx_-pad, cy_-pad, cx_+pad, cy_+pad)

fig, ax = plt.subplots(4, 5, figsize=(13, 10.5))
for col, (s_lo, s_hi) in enumerate(zip(lo, hi)):
    for row, s in [(0, s_lo), (2, s_hi)]:
        im = np.asarray(gens[s].convert('L'), np.float32)
        ax[row, col].imshow(gens[s].crop(box), cmap='gray', vmin=0, vmax=255)
        ax[row, col].set_title(f'seed {s} — det '
                               f'{SW[SW.seed==s].det.iloc[0]:.3f}', fontsize=9)
        ax[row+1, col].imshow((im-BGN)[cy_-pad:cy_+pad, cx_-pad:cx_+pad],
                              cmap='inferno', vmin=0, vmax=90)
        ax[row, col].axis('off'); ax[row+1, col].axis('off')
ax[0,0].text(-0.3, 0.5, 'MISSED', transform=ax[0,0].transAxes,
             rotation=90, va='center', fontsize=13, weight='bold')
ax[2,0].text(-0.3, 0.5, 'FOUND', transform=ax[2,0].transAxes,
             rotation=90, va='center', fontsize=13, weight='bold')
plt.suptitle('rows 1-2: detector scored ~0   rows 3-4: detector scored ~0.9\n'
             'if the missed lesions look like the found ones, these are real failures',
             fontsize=11)
plt.tight_layout(); plt.savefig(OUT/'missed_vs_found.png', dpi=140); plt.show()

In [ ]:
sigma = SW.det.std()          # 0.395, measured on 20 seeds, one cell
for delta in [0.2, 0.3, 0.4, 0.5, 0.7]:
    n = 2 * sigma**2 * (1.96 + 0.84)**2 / delta**2
    print(f'effect {delta}: {int(np.ceil(n))} replicates per cell')
print(f'\nwith 5 replicates you can detect: '
      f'{np.sqrt(2*sigma**2*(1.96+0.84)**2/5):.2f}')